# Lab 4 — Train Your Own Policy

![Training: real experience to replay and World Model learning, then imagination, return targets and Actor-Critic learning in a repeated cycle](figures/training.svg)

Train one DreamerV3 policy in HighwayEnv, inspect its real learning
signals, and watch it drive in Lab 3. The World Model predicts, the
Actor chooses actions, and the Critic estimates value. World Model and
Actor–Critic learning interleave; neither waits for the other to finish.
Training does not guarantee good or safe driving.

## 1. Setup

Follow the **uv** setup in `START_HERE.md` and launch JupyterLab from the
shared environment. Select **Python 3 (ipykernel)** if prompted. Dependencies
are installed once, not inside the notebook.

1. Leave the default settings unchanged and select **Kernel → Restart Kernel and Run All Cells** for a smoke test.
2. Read the evaluation summary and confirm the checkpoint was saved.
3. For a longer run, set `SMOKE_TEST = False`; optionally choose another reward or change one weight.
4. Run All again, then copy the generated `CHECKPOINT` assignment into Lab 3.

On the tested Apple-silicon CPU, a smoke run took about 30–55 seconds
and a full 24,996-step run about 7–8 minutes, plus evaluation. Other
machines may take longer. Allow about 1 GB for dependencies and several
hundred MB for results. Use **Kernel → Interrupt Kernel** to stop;
incomplete runs are retained and cannot be used as completed policies.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path(sys.prefix).resolve().parent
if (sys.version_info[:3] != (3, 11, 12) or
        Path(sys.prefix).resolve() != ROOT / '.venv' or
        not os.environ.get('VIRTUAL_ENV') or
        Path(os.environ['VIRTUAL_ENV']).resolve() != Path(sys.prefix).resolve() or
        not (ROOT / 'pyproject.toml').is_file() or
        not (ROOT / 'inference' / 'lab3_inference.py').is_file() or
        not (ROOT / 'optional_training' / 'train.py').is_file()):
    raise RuntimeError('Wrong kernel. From the extracted lab folder run uv sync --locked, then uv run --locked jupyter lab; select Python 3 (ipykernel) and restart the kernel.')

runtime = ROOT
for name, module in tuple(sys.modules.items()):
    if name.split('.')[0] not in ('dreamerv3', 'embodied', 'optional_training'):
        continue
    locations = list(getattr(module, '__path__', ()))
    if getattr(module, '__file__', None):
        locations.append(module.__file__)
    expected_package = runtime / name.split('.')[0]
    if not locations or any(not Path(p).resolve().is_relative_to(expected_package) for p in locations):
        raise RuntimeError(f'{name} came from another runtime. Restart this notebook kernel before running it.')
sys.path.insert(0, str(runtime))

print('Verified Dreamer Labs kernel:', sys.executable)

## 2. SETTINGS — EDIT HERE

| Reward function | What it encourages | Default weights |
|---|---|---|
| **A — Original HighwayEnv Reward** | Non-collision, speed and right-lane preference; normalized and zero off-road. | Speed 0.4, collision penalty 1.0; right-lane weight 0.1. |
| **C — Speed vs. Collision Reward** | Forward speed, with a collision penalty; no right-lane bonus or normalization. | Speed 1.0, collision penalty 40.0. |

**Step 1:** choose A or C: they are different formulas, not levels of reward strength. **Step 2 (optional):** change either weight; `None` uses that reward's default. Changing a weight does not change the formula. Increasing speed weight favors speed; increasing collision penalty discourages crashes. A's normalization makes the same numbers behave differently from C. Weights must be finite and nonnegative. Both speed bonuses are clipped to 0–1 over 20–30 m/s.

**First run:** leave `SMOKE_TEST = True` to check the pipeline with 1,104 environment steps—not policy quality or a valid reward comparison. **Next:** set it to `False` to train for `TRAINING_STEPS`. To experiment, choose A or C and optionally change one weight for a new run. Inspect the driving evaluation and open the checkpoint in Lab 3. Longer training does not guarantee good driving. Steps are environment steps, not epochs or optimizer updates; the budget must be a multiple of 12 between 1,104 and 24,996.

Lab 4 `SEED` controls training randomness. It need not match Lab 3's driving seed: for example, train with 17 and watch with 1000 or 1001.

### Examples — Try These

**Choosing A or C selects the reward formula. Changing a weight only changes a number within that formula.** Tune either weight under either preset. These examples are not executable cells; edit only the settings cell below.

**1. First run (recommended):** short setup check with the original HighwayEnv reward.
```python
SMOKE_TEST = True
PRESET = "A"
SPEED_WEIGHT = None
COLLISION_PENALTY = None
```
**2. Train with the original reward:** train one policy using A's defaults.
```python
SMOKE_TEST = False
TRAINING_STEPS = 24996
PRESET = "A"
SPEED_WEIGHT = None
COLLISION_PENALTY = None
```
**3. Use a different reward formula:** C selects speed versus collision and its own defaults.
```python
PRESET = "C"
SPEED_WEIGHT = None
COLLISION_PENALTY = None
```
**4. Increase A's speed weight:** A remains A, with speed 0.8 instead of 0.4. This is the historical B reward configuration, not a separate preset.
```python
PRESET = "A"
SPEED_WEIGHT = 0.8
COLLISION_PENALTY = None
```
**5. Reduce C's collision penalty:** C remains C, with penalty 10 instead of 40.
```python
PRESET = "C"
SPEED_WEIGHT = None
COLLISION_PENALTY = 10.0
```
Examples 3–5 change only the listed settings; choose `SMOKE_TEST = False` as well for a longer run.

In [ ]:
# SETTINGS — EDIT HERE
SMOKE_TEST = True
TRAINING_STEPS = 24996

# Step 1: Choose a reward
PRESET = "A"  # "A" or "C"

# Step 2 (optional): Tune its weights
# None = use the chosen reward's default
SPEED_WEIGHT = None
COLLISION_PENALTY = None

SEED = 17
EXPERIMENT_NAME = "my_driver"
OUTPUT_DIR = "results"

## 3. Train Your Policy

This starts one new policy from scratch using the existing trainer.
Progress appears below; optimizer updates begin after replay fills.
Every run gets a unique directory; its eight-character suffix comes from a random UUID, not a checkpoint hash. Interrupted runs are not resumed. The final summary prints the completed directory and a ready-to-copy `CHECKPOINT` line.

In [ ]:
import argparse, json, re, uuid
import pandas as pd
from IPython.display import display

from optional_training.common import load_student_run, seed_value, step_budget, validate_output
from optional_training.rewards import settings
from optional_training.notebook_support import read_metrics, run_process

if type(SMOKE_TEST) is not bool:
    raise ValueError('SMOKE_TEST must be True or False.')
if not SMOKE_TEST:
    if type(TRAINING_STEPS) is not int:
        raise ValueError('TRAINING_STEPS must be an integer.')
    try:
        step_budget(TRAINING_STEPS)
    except argparse.ArgumentTypeError as error:
        raise ValueError('TRAINING_STEPS must be a multiple of 12 between 1,104 and 24,996.') from error
    if TRAINING_STEPS < 1104:
        raise ValueError('TRAINING_STEPS must be at least 1,104 so replay can fill and training can start.')
if type(SEED) is not int:
    raise ValueError('SEED must be an integer between 0 and 4294967295.')
try:
    seed_value(SEED)
except argparse.ArgumentTypeError as error:
    raise ValueError('SEED must be an integer between 0 and 4294967295.') from error
if not isinstance(EXPERIMENT_NAME, str) or not re.fullmatch(r'[A-Za-z0-9_-]{1,64}', EXPERIMENT_NAME):
    raise ValueError('EXPERIMENT_NAME must contain 1–64 letters, numbers, hyphens or underscores.')
defaults = settings(PRESET)['config']
weights = {
    'speed_weight': defaults['high_speed_reward'] if SPEED_WEIGHT is None else SPEED_WEIGHT,
    'collision_penalty': -defaults['collision_reward'] if COLLISION_PENALTY is None else COLLISION_PENALTY,
}
settings(PRESET, weights)
steps = 1104 if SMOKE_TEST else TRAINING_STEPS
eval_seeds = list(range(1000, 1003 if SMOKE_TEST else 1020))
parent = ROOT / Path(OUTPUT_DIR).expanduser()
run_id = f'{EXPERIMENT_NAME}-{uuid.uuid4().hex[:8]}'
run_dir = validate_output(parent / run_id)
eval_dir = validate_output(parent / (run_id + '-evaluation'), (run_dir,))
speed = weights['speed_weight']
penalty = weights['collision_penalty']
print('REWARD CONFIGURATION')
print(f'Reward: {PRESET} — ' + ('Original HighwayEnv' if PRESET == 'A' else 'Speed vs. Collision'))
print(f"Speed weight: {speed} ({'preset default' if SPEED_WEIGHT is None else 'custom'})")
print(f"Collision penalty: {penalty} ({'preset default' if COLLISION_PENALTY is None else 'custom'})")
if PRESET == 'A':
    print(f'Formula: on_road × ({penalty} × non_collision + 0.1 × right_lane + {speed} × speed_bonus) / ({penalty} + 0.1 + {speed})')
else:
    print(f'Formula: reward = {speed} × speed_bonus − {penalty} × collision')
print(f'\nSmoke test: {"ON" if SMOKE_TEST else "OFF"} | Environment steps: {steps:,} | Seed: {SEED}')
if SMOKE_TEST:
    print(f'TRAINING_STEPS={TRAINING_STEPS} is overridden for this installation check.')
print('Run directory:', run_dir, flush=True)
run_process(ROOT / 'optional_training' / 'train.py',
            ['--reward', PRESET, '--steps', steps, '--seed', SEED, '--output', run_dir,
             '--speed-weight', speed, '--collision-penalty', penalty], run_dir)
run_dir, manifest, completed, _, checkpoint = load_student_run(run_dir)
if completed['actual_steps'] != steps or completed['optimizer_updates'] <= 0:
    raise RuntimeError('The completed run has an unexpected step count or no optimizer updates.')
print(f"Completed: {completed['actual_steps']:,} environment steps | {completed['optimizer_updates']:,} optimizer updates")
if SMOKE_TEST:
    print('SMOKE TEST ONLY: this checkpoint is not a full-training result.')

## 4. Observe Learning

These plots use the run's actual logged data. World Model and
Actor/Critic losses are learning signals, not driving scores.
Sparse logs are shown as points, not learning curves. Training return
depends on your chosen reward. Can a lower prediction
loss alone establish that driving is safe?

In [ ]:
print(f"{completed['actual_steps']:,} environment steps | {completed['optimizer_updates']:,} optimizer updates")
if SMOKE_TEST or completed['optimizer_updates'] < 10:
    print('Too few updates to interpret learning progress; curves are suppressed for this short run.')
else:
    import matplotlib.pyplot as plt
    rows = read_metrics(run_dir / 'metrics.jsonl')
    groups = [
        ('World Model loss', [('train/loss/obs', 'Observation'), ('train/loss/rew', 'Reward'),
                              ('train/loss/dyn', 'Dynamics'), ('train/loss/rep', 'Representation')]),
        ('Actor / Critic learning signals', [('train/loss/policy', 'Actor'), ('train/loss/value', 'Critic')]),
        ('Training episode return', [('episode/score', 'Episode return')]),
    ]
    fig, axes = plt.subplots(3, 1, figsize=(10, 10), constrained_layout=True)
    sparse = False
    for ax, (title, series) in zip(axes, groups):
        drawn = False
        for key, label in series:
            points = [(row['step'], row[key]) for row in rows if 'step' in row and key in row]
            if len(points) >= 2:
                if len(points) < 5:
                    ax.scatter(*zip(*points), label=label)
                    sparse = True
                else:
                    ax.plot(*zip(*points), label=label)
                drawn = True
        ax.set(title=title, xlabel='Environment steps', ylabel='Logged value')
        ax.grid(alpha=0.25)
        if drawn:
            ax.legend()
        else:
            ax.text(0.5, 0.5, 'Not enough logged points', ha='center', transform=ax.transAxes)
    plt.show()
    if sparse:
        print('Some series have fewer than five logged points; they are shown as points, not learning curves.')

## 5. Evaluate Your Policy

Evaluate the saved Actor in real simulator episodes using the same
reward configuration as training. Actions are sampled. A smoke test
uses three episodes; a normal experiment uses 20. A smoke run can still
collide because it checks the pipeline, not driving quality. More steps or a
different reward do not guarantee better driving. The action fractions below
are observed sampled-action frequencies across episodes, not the Actor's
per-decision probabilities. If you inspect saved evaluation files, `return_A`
and `return_C` are cross-scores using the fixed default A and C formulas,
not necessarily your selected weights; the displayed episode return uses
your selected training reward.

In [ ]:
run_process(ROOT / 'optional_training' / 'evaluate.py',
            ['--run-dir', run_dir, '--output', eval_dir, '--seeds', *eval_seeds])
summary = json.loads((eval_dir / 'summary.json').read_text())
print('Observed sampled-action frequencies (not Actor probabilities):')
action_names = ['LANE_LEFT', 'IDLE', 'LANE_RIGHT', 'FASTER', 'SLOWER']
display(pd.DataFrame({
    'Action': ['LEFT', 'IDLE', 'RIGHT', 'FASTER', 'SLOWER'],
    'Count': [summary['action_counts'][name] for name in action_names],
    'Fraction': [summary['pooled_action_distribution'][name] for name in action_names],
}).set_index('Action').style.format({'Fraction': '{:.1%}'}))
if SMOKE_TEST:
    print('SMOKE TEST ONLY: three episodes cannot establish policy quality.')

## 6. Watch Your Policy in Lab 3

Lab 4 trains a World Model, Actor and Critic. The checkpoint saves their parameters, other training state and counters; the run directory also holds the matching configuration and reward settings. Lab 3 loads the saved parameter tree and configuration: the Encoder and RSSM estimate the current state from observations and history, then the Actor selects an action. The Critic is not needed to select actions. Only load trusted runs: checkpoints contain Python pickle data. The provided C25K checkpoint is not overwritten.

In [ ]:
run_dir, _, completed, _, _ = load_student_run(run_dir)
metrics = summary['metrics']
if SMOKE_TEST:
    print('SMOKE TEST COMPLETED — The training and evaluation pipeline ran successfully. This short run does not establish that the policy learned good driving behavior or support a comparison between reward presets.')
else:
    print('TRAINING COMPLETED — These evaluation results describe the sampled episodes, not guaranteed driving quality or safety.')
print(f"Environment steps: {completed['actual_steps']:,} | optimizer updates: {completed['optimizer_updates']:,}")
print(f"Evaluation episodes: {summary['episode_count']} | collisions: {summary['collision_count']}/{summary['episode_count']}")
print(f"Mean episode return (this reward): {metrics['own_reward_return']['mean']:.2f} | mean speed: {metrics['mean_speed_mps']['mean']:.2f} m/s | mean forward displacement: {metrics['displacement_m']['mean']:.2f} m")
if not SMOKE_TEST:
    print(f"In these {summary['episode_count']} episodes, {summary['collision_count']} ended in collisions; this describes only the evaluated episodes.")
print('\nYOUR POLICY IS SAVED')
print('Completed run directory:', run_dir)
copy_path = (run_dir.relative_to(ROOT).as_posix() if run_dir.is_relative_to(ROOT / 'results') else
             '~/' + run_dir.relative_to(Path.home()).as_posix() if run_dir.is_relative_to(Path.home()) else str(run_dir))
print('To watch your policy: 1. Open Lab 3.  2. Find SETTINGS — EDIT HERE.')
print('3. Replace CHECKPOINT = None with:')
print('CHECKPOINT = ' + json.dumps(copy_path))
print('4. Run Lab 3 again.')